In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("data/global-data-on-sustainable-energy.csv")
df.head()

In [ ]:
df.dtypes

# TO DO
## Preprocessing
- visualize distribution plots
- drop / impute columns with lots of missing values
- handle density column (object datatype)
- scale

## New variables
- slope of change in renewable energy share over the years (to support classification)

## EDA
- correlation heatmap
- scatter plots
- time trend analysis

## Feature engineering
- derive per capita CO2 emissions (value_co2_emissions_kt_by_country / population*) *population merged from world bank population data
- create classification bins (transition stage based on renewable share slope AND current renewable share): stagnant, transitioning, advanced

## PCA
- drop correlated variables

## Regression & classification models

## Join World Bank population data

The dataset only has total CO2 (`Value_co2_emissions_kt_by_country`) and a **static** density/land area, so dividing CO2 by `Density * Land_Area` gives a per-capita figure that ignores population growth over time.

Below we pull time-varying population from the World Bank API (`SP.POP.TOTL`) and merge it on `(Entity, Year)`.

In [4]:
import requests

year_min, year_max = int(df['Year'].min()), int(df['Year'].max())
url = 'http://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL'
params = {'format': 'json', 'per_page': 20000, 'date': f'{year_min}:{year_max}'}
resp = requests.get(url, params=params, timeout=30)
resp.raise_for_status()
records = resp.json()[1]

pop = pd.DataFrame([
    {'wb_country': r['country']['value'],
     'iso3': r['countryiso3code'],
     'Year': int(r['date']),
     'population': r['value']}
    for r in records if r['value'] is not None
])
print(pop.shape)
pop.head()

(5565, 4)


,wb_country,iso3,Year,population
0,Africa Eastern and Southern,AFE,2020,694446100
1,Africa Eastern and Southern,AFE,2019,675950189
2,Africa Eastern and Southern,AFE,2018,657801085
3,Africa Eastern and Southern,AFE,2017,640058741
4,Africa Eastern and Southern,AFE,2016,623369401


In [7]:
# World Bank uses different names for some countries than this dataset's `Entity` column.
# Map WB names -> Entity names so the merge keys line up.
wb_to_entity = {
    'Bahamas, The': 'Bahamas',
    'Brunei Darussalam': 'Brunei',
    'Cabo Verde': 'Cape Verde',
    "Cote d'Ivoire": "Cote d'Ivoire",
    'Congo, Dem. Rep.': 'Democratic Republic of Congo',
    'Congo, Rep.': 'Congo',
    'Egypt, Arab Rep.': 'Egypt',
    'Gambia, The': 'Gambia',
    'Hong Kong SAR, China': 'Hong Kong',
    'Iran, Islamic Rep.': 'Iran',
    "Korea, Dem. People's Rep.": 'North Korea',
    'Korea, Rep.': 'South Korea',
    'Kyrgyz Republic': 'Kyrgyzstan',
    'Lao PDR': 'Laos',
    'Macao SAR, China': 'Macao',
    'Micronesia, Fed. Sts.': 'Micronesia (country)',
    'Russian Federation': 'Russia',
    'Slovak Republic': 'Slovakia',
    'St. Kitts and Nevis': 'Saint Kitts and Nevis',
    'St. Lucia': 'Saint Lucia',
    'St. Vincent and the Grenadines': 'Saint Vincent and the Grenadines',
    'Syrian Arab Republic': 'Syria',
    'Turkiye': 'Turkey',
    'Venezuela, RB': 'Venezuela',
    'Viet Nam': 'Vietnam',
    'Yemen, Rep.': 'Yemen',
}
pop['Entity'] = pop['wb_country'].replace(wb_to_entity)

# Drop population if it already exists, so re-running this cell is safe
df = df.drop(columns=['population'], errors='ignore')
df = df.merge(pop[['Entity', 'Year', 'population']], on=['Entity', 'Year'], how='left')

# Check match quality
n_missing = df['population'].isna().sum()
unmatched = sorted(df.loc[df['population'].isna(), 'Entity'].unique())
print(f'rows without population: {n_missing} / {len(df)} ({n_missing/len(df):.1%})')
print(f'unmatched entities ({len(unmatched)}):', unmatched)

rows without population: 43 / 3649 (1.2%)
unmatched entities (3): ['French Guiana', 'Puerto Rico', 'Somalia']


In [8]:
# CO2 column is in kilotons -> convert to tonnes, then divide by population for tonnes per person
df['co2_per_capita_t'] = df['Value_co2_emissions_kt_by_country'] * 1000 / df['population']

# Sanity check: latest year WITH CO2 data per country
# (the most recent year of the dataset has NaN CO2, so filter that out first)
# Expected ballparks: USA ~15, China ~7, India ~2, Qatar 30+, Germany ~8 t/person
sanity = ['United States', 'China', 'India', 'Qatar', 'Germany']
(df[df['Entity'].isin(sanity) & df['co2_per_capita_t'].notna()]
   .sort_values(['Entity', 'Year'])
   .groupby('Entity').tail(1)
   [['Entity', 'Year', 'Value_co2_emissions_kt_by_country', 'population', 'co2_per_capita_t']])

,Entity,Year,Value_co2_emissions_kt_by_country,population,co2_per_capita_t
733,China,2019,1.070722e+07,1.407745e+09,7.605937
1301,Germany,2019,6.574000e+05,8.309296e+07,7.911621
1553,India,2019,2.456300e+06,1.389030e+09,1.768356
2722,Qatar,2019,9.197000e+04,2.638657e+06,34.854853
3521,United States,2019,4.817720e+06,3.302262e+08,14.589151
